In [121]:
using LowLevelFEM, LinearAlgebra

In [122]:
openGeometry("rectangles.geo")

In [123]:
#openPreProcessor()

In [124]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=2, fieldName=:u);

In [125]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0)
bc_top = BoundaryCondition("top", ux=0, uy=(x,y,z)->x*(x-10)/130)

K = ∫(SymGrad(U) ⋅ D(:PlaneStress, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0])

u = solveField(K, f, support=[bc_bottom, bc_top])

#showDoFResults(u, name="u", factor=1, visible=true)

nodal VectorField
[0.0; 0.0; … ; -0.04806520265106004; -0.11747272556448052;;]


## Penalty contact

The contact object contains only the contact geometry and kinematics. The full
contact operator maps the displacement field to the local contact space,

$$
G: V_u \rightarrow V_c ,
$$

while $P_a$ selects the currently active contact nodes,

$$
G_a=P_aG.
$$

The penalty surface operator is assembled once with the ordinary LLFEM weak-form
machinery and then restricted to the slave surface. In 2D the contact-space
ordering is $(n,t)$. For frictionless contact $c_t=0$.


In [126]:

r = nodePositionVector(U)

C = contact(
    u,
    master="master",
    slave="slave",
    topology_tol=0.01
)

cn = 1e7
ct = 0.0

Dc = [cn 0.0
      0.0 ct]

C0 = ∫(U ⋅ Dc ⋅ U, Γ="slave")
Cc = subSystemMatrix(C0; onPhysicalGroup="slave");



At a fixed contact geometry the active penalty contribution is

$$
d_a=P_a d,\qquad
C_a=P_a C_c P_a^T,\qquad
G_a=P_aG,
$$

$$
r_c=G_a^T C_a d_a,\qquad
K_c=G_a^T C_aG_a.
$$

Since $d=G(r+u)$, freezing the current geometry gives the Newton equation

$$
(K+K_c)u_{\mathrm{new}}=f-K_cr.
$$

Thus the linear correction can still be solved with the ordinary `solveField`
function. A line search is retained because the projection and active set may
change during the iteration.


In [127]:

support = [bc_bottom, bc_top]

support_increment = [
    BoundaryCondition("bottom", ux=0, uy=0),
    BoundaryCondition("top",    ux=0, uy=0)
]

free = freeDoFs(U, support)

# Only used to compare nonlinear residuals on unconstrained DoFs.
freeNorm(v::VectorField) = LinearAlgebra.norm(elementsToNodes(v).a[free, 1])

u_it = copy(u)

old_tags = copy(C.master_element_tags)
old_G = copy(C.G)

for iter in 1:30

    updateContact!(C, u_it)

    nchanged = count(old_tags .!= C.master_element_tags)

    dG = norm(C.G - old_G) /
         max(norm(old_G), eps())

    old_tags = copy(C.master_element_tags)
    old_G = copy(C.G)

    # Active contact algebra
    Ga = C.Pa * C.G
    Ca = C.Pa * Cc * C.Pa'
    da = C.Pa * C.d

    # Contact residual and frozen-geometry tangent
    rc = Ga' * (Ca * da)
    Kc = Ga' * Ca * Ga

    # Total residual
    R = K * u_it - f + rc
    R0 = freeNorm(R)

    # Full frozen-geometry Newton step:
    # (K + Kc) u_new = f - Kc r
    Δu = solveField(
        K + Kc,
        -R,
        support=support_increment
    )

    # Line search because G, projection and the active set change with u.
    α = 1.0
    u_trial = copy(u_it)
    Rtrial = R

    while α > 1e-6

        u_trial = u_it + α * Δu

        updateContact!(C, u_trial)

        Ga_trial = C.Pa * C.G
        Ca_trial = C.Pa * Cc * C.Pa'
        da_trial = C.Pa * C.d

        rc_trial = Ga_trial' * (Ca_trial * da_trial)

        Rtrial = K * u_trial - f + rc_trial

        freeNorm(Rtrial) < R0 && break

        α *= 0.5
    end

    u_it = u_trial

    err = freeNorm(α * Δu) /
          max(freeNorm(u_it), eps())

    println(
        "iter = ", iter,
        ", α = ", α,
        ", active = ", count(C.active),
        ", master changes = ", nchanged,
        ", dG = ", dG,
        ", min gap = ", minimum(C.gap_values),
        ", error = ", err,
        ", |R| = ", freeNorm(Rtrial)
    )

    err < 1e-8 && break
end

u = u_it

# Synchronize the stored contact state with the converged field.
updateContact!(C, u);


iter = 1, α = 1.0, active = 31, master changes = 0, dG = 0.0, min gap = -0.00047174113054511353, error = 0.20717588720843208, |R| = 533.8267474181444
iter = 2, α = 1.0, active = 29, master changes = 0, dG = 0.01058305233987416, min gap = -0.00047168224038326105, error = 0.004384993810851824, |R| = 27.953390415529395
iter = 3, α = 1.0, active = 29, master changes = 0, dG = 0.004034205403803652, min gap = -0.0004715118131172631, error = 9.289010412985201e-5, |R| = 0.078472435439771
iter = 4, α = 1.0, active = 29, master changes = 0, dG = 4.14963532644938e-5, min gap = -0.00047151364332827976, error = 1.9294191738721446e-6, |R| = 0.0035280560716072994
iter = 5, α = 1.0, active = 29, master changes = 0, dG = 2.6731491656480406e-6, min gap = -0.0004715134083756505, error = 8.178875162760836e-8, |R| = 0.00013536993958554854
iter = 6, α = 1.0, active = 29, master changes = 0, dG = 7.560436955983603e-8, min gap = -0.00047151341572610096, error = 3.598014452295549e-9, |R| = 7.1194413293225445e-

In [128]:

showDoFResults(u, name="u", factor=1, visible=true)


0


## Contact fields

`C.d` is a reduced `ContactVector`. Mapping it back to the displacement mesh
makes the local contact components available through the ordinary field API.

For the current closest-point geometry, `D[1]` is the normal gap. The tangential
component of the current position difference is approximately zero by
construction; tangential slip will later be accumulated from displacement
increments/history.


In [129]:

Dn = VectorField(C.d)

gap = Dn[1]

# Active normal gap, expanded back to the full contact space.
da_full = C.Pa' * (C.Pa * C.d)
Da = VectorField(da_full)

# Pointwise penalty traction (positive in compression).
pressure = -cn * Da[1]

gap

plotOnBeam("slave", nodesToElements(pressure))
plotOnBeam("slave", nodesToElements(gap))


2

In [130]:

# Example postprocessing:
# showElementResults(nodesToElements(gap), name="gap", visible=true)
# showElementResults(nodesToElements(pressure), name="pressure", visible=true)

openPostProcessor()
